# 🚆 ETL — Données ferroviaires européennes

**Pipeline complet : Extract → Transform → Load**

| Fichier | Contenu |
|---|---|
| `co2_comparaison_europe.csv` | Émissions CO₂ par mode de transport par pays |
| `france_rail_OD_flows.csv` | Flux OD ferroviaires France |
| `germany_rail_OD_flows.csv` | Flux OD ferroviaires Allemagne |
| `italy_rail_OD_flows.csv` | Flux OD ferroviaires Italie |
| `spain_rail_OD_flows.csv` | Flux OD ferroviaires Espagne |
| `switzerland_rail_OD_flows.csv` | Flux OD ferroviaires Suisse |
| `cross_border_rail_OD_IT_DE_CH_ES.csv` | Flux transfrontaliers (IT, DE, CH, ES) |

---
## 0. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── Chemins (adapter si besoin) ──────────────────────────────────
DATA_DIR   = "."        # dossier contenant les CSV sources
OUTPUT_DIR = "./etl_output" # dossier de sortie
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print('✅ Prêt')

✅ Prêt


---
## 1. EXTRACT — Chargement des fichiers bruts

In [ ]:
def load_csv(filename, sep=';'):
    """Charge un CSV et affiche un résumé rapide."""
    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path, sep=sep, encoding='utf-8-sig')
    print(f'📂  {filename:<50}  {df.shape[0]:>5} lignes | {df.shape[1]} colonnes')
    return df

print('Chargement des fichiers...\n')
df_co2        = load_csv('co2_comparaison_europe.csv')
df_france     = load_csv('trajets/france_rail_OD_flows.csv')
df_germany    = load_csv('trajets/germany_rail_OD_flows.csv')
df_italy      = load_csv('trajets/italy_rail_OD_flows.csv')
df_spain      = load_csv('trajets/spain_rail_OD_flows.csv')
df_switz      = load_csv('trajets/switzerland_rail_OD_flows.csv')
df_portugal   = load_csv('trajets/portugal_rail_OD_flows.csv')
df_crossbord  = load_csv('trajets/cross_border_rail_OD_IT_DE_FR_CH_ES_PT_bidirectional.csv')
print('\n✅ Tous les fichiers chargés.')

Chargement des fichiers...

📂  co2_comparaison_europe.csv                              5 lignes | 7 colonnes
📂  rails_OD_flows/france_rail_OD_flows.csv              2160 lignes | 10 colonnes
📂  rails_OD_flows/germany_rail_OD_flows.csv              110 lignes | 10 colonnes
📂  rails_OD_flows/italy_rail_OD_flows.csv                 95 lignes | 10 colonnes
📂  rails_OD_flows/spain_rail_OD_flows.csv                 83 lignes | 10 colonnes
📂  rails_OD_flows/switzerland_rail_OD_flows.csv           88 lignes | 10 colonnes
📂  rails_OD_flows/portugal_rail_OD_flows.csv             138 lignes | 10 colonnes
📂  rails_OD_flows/cross_border_rail_OD_IT_DE_FR_CH_ES_PT_bidirectional.csv    396 lignes | 12 colonnes

✅ Tous les fichiers chargés.


---
## 2. EXPLORE — Aperçu des données brutes

In [4]:
print('=== CO2 ===')
display(df_co2)

print('\n=== Flux France (5 premières lignes) ===')
display(df_france.head())

print('\n=== Flux Allemagne (5 premières lignes) ===')
display(df_germany.head())

print('\n=== Flux cross-border (5 premières lignes) ===')
display(df_crossbord.head())

=== CO2 ===


,Pays,Emissions_Train_g_km,Emissions_Voiture_g_km,Emissions_Avion_g_km,CO2_evite_Train_vs_Voiture_g_km,CO2_evite_Train_vs_Avion_g_km,Reduction_Train_vs_Voiture_%
0,France,3.0,192,255,189.0,252.0,98.44
1,Suisse,1.5,180,255,178.5,253.5,99.17
2,Portugal,30.0,190,250,160.0,220.0,84.21
3,Allemagne,32.0,192,255,160.0,223.0,83.33
4,Italie,35.0,185,250,150.0,215.0,81.08



=== Flux France (5 premières lignes) ===


,year,origin_station_name,destination_station_name,origin_city,destination_city,origin_region,destination_region,Passengers_millions,Type,source
0,2016,Paris Gare du Nord,Paris Gare de Lyon,Paris,Paris,Île-de-France,Île-de-France,1.05,TGV/Intercités,"SNCF Open Data - Fréquentation en gares, SNCF rapports a..."
1,2016,Paris Gare du Nord,Paris Saint-Lazare,Paris,Paris,Île-de-France,Île-de-France,1.13,TGV/Intercités,"SNCF Open Data - Fréquentation en gares, SNCF rapports a..."
2,2016,Paris Gare du Nord,Paris Montparnasse,Paris,Paris,Île-de-France,Île-de-France,0.88,TGV/Intercités,"SNCF Open Data - Fréquentation en gares, SNCF rapports a..."
3,2016,Paris Gare du Nord,Paris Est,Paris,Paris,Île-de-France,Île-de-France,0.93,TGV/Intercités,"SNCF Open Data - Fréquentation en gares, SNCF rapports a..."
4,2016,Paris Gare du Nord,Paris Austerlitz,Paris,Paris,Île-de-France,Île-de-France,0.89,TGV/Intercités,"SNCF Open Data - Fréquentation en gares, SNCF rapports a..."



=== Flux Allemagne (5 premières lignes) ===


,year,origin_station_name,destination_station_name,origin_city,destination_city,origin_region,destination_region,Passengers_millions,Type,source
0,2016,Berlin Hbf,Frankfurt (Main) Hbf,Berlin,Frankfurt am Main,Berlin,Hessen,2.539,Fernverkehr,Eurostat - Railway passenger transport statistics (used ...
1,2016,Berlin Hbf,Hannover Hbf,Berlin,Hannover,Berlin,Niedersachsen,1.098,Fernverkehr,Deutsche Bahn - Facts & Figures (long-distance and regio...
2,2016,Hamburg Hbf,Frankfurt (Main) Hbf,Hamburg,Frankfurt am Main,Hamburg,Hessen,2.493,Fernverkehr,Eurostat - Railway passenger transport statistics (used ...
3,2016,Hamburg Hbf,Köln Hbf,Hamburg,Köln,Hamburg,Nordrhein-Westfalen,0.848,Fernverkehr,VDV - Statistik zum SPNV in Deutschland (used for calibr...
4,2016,Hamburg Hbf,Leipzig Hbf,Hamburg,Leipzig,Hamburg,Sachsen,1.139,Fernverkehr,Deutsche Bahn - Facts & Figures (long-distance and regio...



=== Flux cross-border (5 premières lignes) ===


,year,origin_country,origin_station_name,origin_city,origin_region,destination_country,destination_station_name,destination_city,destination_region,Passengers_millions,Type,source
0,2016,Italy,Milano Centrale,Milano,Lombardia,Germany,München Hbf,München,Bayern,1.918,Long-distance,More and more trains crossing European borders - Europea...
1,2016,Italy,Bolzano,Bolzano,Trentino-Alto Adige,Germany,München Hbf,München,Bayern,2.077,Long-distance,Rail operators international route information (SNCF / R...
2,2016,Italy,Milano Centrale,Milano,Lombardia,Switzerland,Zürich HB,Zürich,Zürich,0.705,Long-distance,Rail operators international route information (SNCF / R...
3,2016,Italy,Milano Centrale,Milano,Lombardia,Switzerland,Lugano,Lugano,Ticino,0.258,Long-distance,More and more trains crossing European borders - Europea...
4,2016,Italy,Domodossola,Domodossola,Piemonte,Switzerland,Brig,Brig,Valais,0.158,Long-distance,More and more trains crossing European borders - Europea...


In [5]:
# Résumé qualité des données brutes
datasets = {
    'co2'          : df_co2,
    'france'       : df_france,
    'germany'      : df_germany,
    'italy'        : df_italy,
    'spain'        : df_spain,
    'switzerland'  : df_switz,
    'portugal'     : df_portugal,
    'cross_border' : df_crossbord
}

print(f"{'Dataset':<16} {'Lignes':>6} {'Colonnes':>9} {'Manquants':>10}")
print('-' * 46)
for name, df in datasets.items():
    missing = df.isnull().sum().sum()
    print(f"{name:<16} {df.shape[0]:>6} {df.shape[1]:>9} {missing:>10}")

Dataset          Lignes  Colonnes  Manquants
----------------------------------------------
co2                   5         7          0
france             2160        10          0
germany             110        10          0
italy                95        10          0
spain                83        10          0
switzerland          88        10          0
portugal            138        10          0
cross_border        396        12          0


---
## 3. TRANSFORM — Nettoyage & standardisation

### 3.1 Table CO₂

In [6]:
def transform_co2(df):
    df = df.copy()
    df.columns = [
        'country', 'emissions_train_g_km', 'emissions_car_g_km',
        'emissions_plane_g_km', 'co2_saved_vs_car_g_km',
        'co2_saved_vs_plane_g_km', 'reduction_vs_car_pct'
    ]
    df['country'] = df['country'].str.strip()
    num_cols = df.columns[1:]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    print(f'✅ CO2 transformé : {df.shape}')
    return df

df_co2_clean = transform_co2(df_co2)
display(df_co2_clean)

✅ CO2 transformé : (5, 7)


,country,emissions_train_g_km,emissions_car_g_km,emissions_plane_g_km,co2_saved_vs_car_g_km,co2_saved_vs_plane_g_km,reduction_vs_car_pct
0,France,3.0,192,255,189.0,252.0,98.44
1,Suisse,1.5,180,255,178.5,253.5,99.17
2,Portugal,30.0,190,250,160.0,220.0,84.21
3,Allemagne,32.0,192,255,160.0,223.0,83.33
4,Italie,35.0,185,250,150.0,215.0,81.08


### 3.2 Flux OD nationaux (France, Allemagne, Italie, Espagne, Suisse, Portugal)

In [16]:
# Mapping pays → (code ISO, label anglais, label CO2)
COUNTRY_MAP = {
    'france'      : ('FR', 'France',      'France'),
    'germany'     : ('DE', 'Germany',     'Allemagne'),
    'italy'       : ('IT', 'Italy',       'Italie'),
    'spain'       : ('ES', 'Spain',       'Espagne'),
    'switzerland' : ('CH', 'Switzerland', 'Suisse'),
    'portugal'    : ('PT', 'Portugal',    'Portugal')
}

# Colonnes standardisées communes aux 5 fichiers nationaux
NATIONAL_COLS = [
    'year', 'origin_station', 'destination_station',
    'origin_city', 'destination_city',
    'origin_region', 'destination_region',
    'passengers_millions', 'service_type', 'source'
]

def transform_od_national(df, country_key):
    """Nettoyage générique pour les flux OD nationaux."""
    df = df.copy()
    iso, label, _ = COUNTRY_MAP[country_key]

    df.columns = NATIONAL_COLS

    # Métadonnées pays
    df['country']     = label
    df['country_iso'] = iso
    df['flow_type']   = 'domestic'

    # Cast types
    df['year']               = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df['passengers_millions'] = pd.to_numeric(df['passengers_millions'], errors='coerce')

    # Nettoyage textes
    text_cols = ['origin_station', 'destination_station',
                 'origin_city', 'destination_city',
                 'origin_region', 'destination_region']
    for col in text_cols:
        df[col] = df[col].str.strip()

    # Suppression doublons
    before = len(df)
    df.drop_duplicates(subset=['year', 'origin_station', 'destination_station'], inplace=True)
    removed = before - len(df)
    if removed:
        print(f'  ⚠️  {removed} doublon(s) supprimé(s) dans {label}')

    # Suppression lignes sans passagers
    df.dropna(subset=['passengers_millions'], inplace=True)

    print(f'✅ {label:<15} transformé : {df.shape[0]} lignes')
    return df

df_france_clean  = transform_od_national(df_france,  'france')
df_germany_clean = transform_od_national(df_germany, 'germany')
df_italy_clean   = transform_od_national(df_italy,   'italy')
df_spain_clean   = transform_od_national(df_spain,   'spain')
df_switz_clean   = transform_od_national(df_switz,   'switzerland')
df_portugal_clean= transform_od_national(df_portugal,'portugal')

✅ France          transformé : 2160 lignes
✅ Germany         transformé : 110 lignes
✅ Italy           transformé : 95 lignes
✅ Spain           transformé : 83 lignes
✅ Switzerland     transformé : 88 lignes
✅ Portugal        transformé : 138 lignes


### 3.3 Flux cross-border

In [8]:
def transform_crossborder(df):
    df = df.copy()
    df.columns = [
        'year',
        'origin_country_iso', 'origin_station', 'origin_city', 'origin_region',
        'destination_country_iso', 'destination_station', 'destination_city', 'destination_region',
        'passengers_millions', 'service_type', 'source'
    ]
    df['flow_type'] = 'cross_border'

    df['year']                = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df['passengers_millions'] = pd.to_numeric(df['passengers_millions'], errors='coerce')

    text_cols = ['origin_station', 'destination_station',
                 'origin_city', 'destination_city',
                 'origin_region', 'destination_region']
    for col in text_cols:
        df[col] = df[col].str.strip()

    # Normaliser les noms de pays ISO (ex: 'Italy' → 'IT')
    country_to_iso = {
        'France': 'FR', 'Germany': 'DE', 'Italy': 'IT',
        'Spain': 'ES', 'Switzerland': 'CH', 'Portugal': 'PT'
    }
    df['origin_country_iso']      = df['origin_country_iso'].map(country_to_iso).fillna(df['origin_country_iso'])
    df['destination_country_iso'] = df['destination_country_iso'].map(country_to_iso).fillna(df['destination_country_iso'])

    df.dropna(subset=['passengers_millions'], inplace=True)
    print(f'✅ Cross-border transformé : {df.shape[0]} lignes')
    return df

df_crossbord_clean = transform_crossborder(df_crossbord)
display(df_crossbord_clean.head())

✅ Cross-border transformé : 396 lignes


,year,origin_country_iso,origin_station,origin_city,origin_region,destination_country_iso,destination_station,destination_city,destination_region,passengers_millions,service_type,source,flow_type
0,2016,IT,Milano Centrale,Milano,Lombardia,DE,München Hbf,München,Bayern,1.918,Long-distance,More and more trains crossing European borders - Europea...,cross_border
1,2016,IT,Bolzano,Bolzano,Trentino-Alto Adige,DE,München Hbf,München,Bayern,2.077,Long-distance,Rail operators international route information (SNCF / R...,cross_border
2,2016,IT,Milano Centrale,Milano,Lombardia,CH,Zürich HB,Zürich,Zürich,0.705,Long-distance,Rail operators international route information (SNCF / R...,cross_border
3,2016,IT,Milano Centrale,Milano,Lombardia,CH,Lugano,Lugano,Ticino,0.258,Long-distance,More and more trains crossing European borders - Europea...,cross_border
4,2016,IT,Domodossola,Domodossola,Piemonte,CH,Brig,Brig,Valais,0.158,Long-distance,More and more trains crossing European borders - Europea...,cross_border


---
## 4. CONSOLIDATION — Table OD unifiée

In [9]:
# Schéma commun minimal pour le concat
COMMON_COLS = [
    'year', 'origin_station', 'destination_station',
    'origin_city', 'destination_city',
    'origin_region', 'destination_region',
    'passengers_millions', 'service_type', 'flow_type', 'source'
]

# Flux nationaux : on ajoute origin/destination_country_iso identiques
national_frames = []
for df, iso in [
    (df_france_clean,  'FR'),
    (df_germany_clean, 'DE'),
    (df_italy_clean,   'IT'),
    (df_spain_clean,   'ES'),
    (df_switz_clean,   'CH'),
    (df_portugal_clean,'PT')
]:
    tmp = df[COMMON_COLS].copy()
    tmp['origin_country_iso']      = iso
    tmp['destination_country_iso'] = iso
    national_frames.append(tmp)

# Flux cross-border
cb = df_crossbord_clean[COMMON_COLS + ['origin_country_iso', 'destination_country_iso']].copy()

# Concatenation
df_all_od = pd.concat(national_frames + [cb], ignore_index=True)

print(f'✅ Table OD consolidée : {df_all_od.shape[0]:,} lignes | {df_all_od.shape[1]} colonnes')
display(df_all_od.sample(5, random_state=42))

✅ Table OD consolidée : 3,070 lignes | 13 colonnes


,year,origin_station,destination_station,origin_city,destination_city,origin_region,destination_region,passengers_millions,service_type,flow_type,source,origin_country_iso,destination_country_iso
1446,2022,Paris Gare du Nord,Lyon Perrache,Paris,Lyon,Île-de-France,Auvergne-Rhône-Alpes,7.660,TGV/Intercités,domestic,"SNCF Open Data - Fréquentation en gares, SNCF rapports a...",FR,FR
2595,2019,Braga,Faro,Braga,Faro,Braga,Faro,1.347,Regional / Urban,domestic,CP - Comboios de Portugal: relatórios de tráfego e mobil...,PT,PT
1644,2022,Nantes,Lille Flandres,Nantes,Lille,Pays de la Loire,Hauts-de-France,1.080,TER/Intercités,domestic,"SNCF Open Data - Fréquentation en gares, SNCF rapports a...",FR,FR
194,2016,Toulouse Matabiau,Strasbourg,Toulouse,Strasbourg,Occitanie,Grand Est,1.130,TER/Intercités,domestic,"SNCF Open Data - Fréquentation en gares, SNCF rapports a...",FR,FR
240,2017,Paris Gare du Nord,Paris Gare de Lyon,Paris,Paris,Île-de-France,Île-de-France,0.910,TGV/Intercités,domestic,"SNCF Open Data - Fréquentation en gares, SNCF rapports a...",FR,FR


### 4.1 Enrichissement — jointure avec CO₂

In [10]:
# Mapping ISO → nom dans la table CO2 (noms en français)
ISO_TO_CO2_COUNTRY = {
    'FR': 'France',
    'DE': 'Allemagne',
    'IT': 'Italie',
    'ES': 'Espagne',
    'CH': 'Suisse',
    'PT': 'Portugal'
}

df_all_od['co2_country_key'] = df_all_od['origin_country_iso'].map(ISO_TO_CO2_COUNTRY)

df_enriched = df_all_od.merge(
    df_co2_clean[['country', 'emissions_train_g_km', 'emissions_car_g_km',
                  'co2_saved_vs_car_g_km', 'reduction_vs_car_pct']],
    left_on='co2_country_key',
    right_on='country',
    how='left'
).drop(columns=['country', 'co2_country_key'])

print(f'✅ Table enrichie : {df_enriched.shape}')
print(f"   Taux de match CO2 : {df_enriched['emissions_train_g_km'].notna().mean():.1%}")
display(df_enriched.head())

✅ Table enrichie : (3070, 17)
   Taux de match CO2 : 95.5%


,year,origin_station,destination_station,origin_city,destination_city,origin_region,destination_region,passengers_millions,service_type,flow_type,source,origin_country_iso,destination_country_iso,emissions_train_g_km,emissions_car_g_km,co2_saved_vs_car_g_km,reduction_vs_car_pct
0,2016,Paris Gare du Nord,Paris Gare de Lyon,Paris,Paris,Île-de-France,Île-de-France,1.05,TGV/Intercités,domestic,"SNCF Open Data - Fréquentation en gares, SNCF rapports a...",FR,FR,3.0,192.0,189.0,98.44
1,2016,Paris Gare du Nord,Paris Saint-Lazare,Paris,Paris,Île-de-France,Île-de-France,1.13,TGV/Intercités,domestic,"SNCF Open Data - Fréquentation en gares, SNCF rapports a...",FR,FR,3.0,192.0,189.0,98.44
2,2016,Paris Gare du Nord,Paris Montparnasse,Paris,Paris,Île-de-France,Île-de-France,0.88,TGV/Intercités,domestic,"SNCF Open Data - Fréquentation en gares, SNCF rapports a...",FR,FR,3.0,192.0,189.0,98.44
3,2016,Paris Gare du Nord,Paris Est,Paris,Paris,Île-de-France,Île-de-France,0.93,TGV/Intercités,domestic,"SNCF Open Data - Fréquentation en gares, SNCF rapports a...",FR,FR,3.0,192.0,189.0,98.44
4,2016,Paris Gare du Nord,Paris Austerlitz,Paris,Paris,Île-de-France,Île-de-France,0.89,TGV/Intercités,domestic,"SNCF Open Data - Fréquentation en gares, SNCF rapports a...",FR,FR,3.0,192.0,189.0,98.44


---
## 5. CONTRÔLES QUALITÉ & STATISTIQUES

In [11]:
print('=== Répartition par pays et type de flux ===')
display(
    df_enriched.groupby(['origin_country_iso', 'flow_type'])
               .agg(
                   nb_routes          = ('passengers_millions', 'count'),
                   total_passengers_M = ('passengers_millions', 'sum'),
               )
               .round(2)
               .reset_index()
               .sort_values('total_passengers_M', ascending=False)
)

=== Répartition par pays et type de flux ===


,origin_country_iso,flow_type,nb_routes,total_passengers_M
7,FR,domestic,2160,6764.81
3,DE,domestic,110,291.97
9,IT,domestic,95,288.42
5,ES,domestic,83,259.88
1,CH,domestic,88,252.48
11,PT,domestic,138,167.40
6,FR,cross_border,126,159.48
8,IT,cross_border,72,96.32
0,CH,cross_border,63,70.12
2,DE,cross_border,45,56.95


In [12]:
print('=== Top 10 des routes les plus fréquentées ===')
top10 = (
    df_enriched
    .groupby(['origin_country_iso', 'origin_city', 'destination_city'])
    ['passengers_millions'].sum()
    .reset_index()
    .sort_values('passengers_millions', ascending=False)
    .head(10)
)
display(top10)

=== Top 10 des routes les plus fréquentées ===


,origin_country_iso,origin_city,destination_city,passengers_millions
218,FR,Lyon,Paris,813.68
246,FR,Paris,Lyon,805.06
245,FR,Paris,Lille,708.41
207,FR,Lille,Paris,704.18
247,FR,Paris,Marseille,308.16
227,FR,Marseille,Paris,306.82
241,FR,Paris,Bordeaux,254.82
251,FR,Paris,Paris,251.49
197,FR,Bordeaux,Paris,249.36
252,FR,Paris,Rennes,203.24


In [13]:
print('=== Évolution totale par année ===')
display(
    df_enriched.groupby('year')
               .agg(
                   nb_routes          = ('passengers_millions', 'count'),
                   total_passengers_M = ('passengers_millions', 'sum'),
               )
               .round(2)
)

=== Évolution totale par année ===


,nb_routes,total_passengers_M
year,,
2016,352,1024.29
2017,348,993.61
2018,345,1021.33
2019,335,993.31
2020,335,491.12
2021,346,815.21
2022,323,984.62
2023,341,1077.17
2024,345,1103.55


In [14]:
print('=== Comparatif CO₂ par pays (train vs voiture) ===')
display(df_co2_clean[['country', 'emissions_train_g_km', 'emissions_car_g_km',
                       'reduction_vs_car_pct']].sort_values('emissions_train_g_km'))

=== Comparatif CO₂ par pays (train vs voiture) ===


,country,emissions_train_g_km,emissions_car_g_km,reduction_vs_car_pct
1,Suisse,1.5,180,99.17
0,France,3.0,192,98.44
2,Portugal,30.0,190,84.21
3,Allemagne,32.0,192,83.33
4,Italie,35.0,185,81.08


---
## 6. LOAD — Export des tables finales

In [15]:
# ── Table 1 : flux OD enrichis (table principale) ────────────────
out_od = os.path.join(OUTPUT_DIR, 'od_flows_enriched.csv')
df_enriched.to_csv(out_od, index=False, sep=';', encoding='utf-8-sig')
print(f'💾 od_flows_enriched.csv     → {len(df_enriched):,} lignes')

# ── Table 2 : CO2 nettoyé ─────────────────────────────────────────
out_co2 = os.path.join(OUTPUT_DIR, 'co2_clean.csv')
df_co2_clean.to_csv(out_co2, index=False, sep=';', encoding='utf-8-sig')
print(f'💾 co2_clean.csv             → {len(df_co2_clean):,} lignes')

# ── Table 3 : agrégat par pays/année/type ─────────────────────────
df_agg = (
    df_enriched
    .groupby(['year', 'origin_country_iso', 'flow_type'])
    .agg(
        nb_routes          = ('passengers_millions', 'count'),
        total_passengers_M = ('passengers_millions', 'sum'),
        avg_passengers_M   = ('passengers_millions', 'mean'),
        max_route_M        = ('passengers_millions', 'max'),
    )
    .round(3)
    .reset_index()
)
out_agg = os.path.join(OUTPUT_DIR, 'od_aggregated.csv')
df_agg.to_csv(out_agg, index=False, sep=';', encoding='utf-8-sig')
print(f'💾 od_aggregated.csv         → {len(df_agg):,} lignes')

print(f'\n✅ Export terminé — dossier : {os.path.abspath(OUTPUT_DIR)}')

💾 od_flows_enriched.csv     → 3,070 lignes
💾 co2_clean.csv             → 5 lignes
💾 od_aggregated.csv         → 108 lignes

✅ Export terminé — dossier : c:\Users\rejom\Documents\github\MsprDS\data\output


---
## 7. Récapitulatif du pipeline

| Étape | Description | Résultat |
|---|---|---|
| **Extract** | Lecture de 7 CSV (séparateur `;`, encoding `utf-8-sig`) | ~2 500 lignes brutes |
| **Transform** | Renommage colonnes, cast types, suppression doublons/NaN, normalisation ISO | Schéma unifié |
| **Consolidation** | Concat des 5 pays + cross-border + jointure CO₂ | 1 table enrichie |
| **Load** | Export en 3 CSV structurés dans `./output/` | Prêt pour analyse / BI |